# 第 3 周练习 — 本地开源模型

第 3 周是关于拥抱面孔：**标记器**、**管道**和运行 **开源
型号**。 Day-5 会议记录项目在 Colab GPU 上使用 Whisper + Llama-8B；这里
是一个在 CPU 上本地运行的最小版本 - 比较分词器，然后将
使用开源摘要器将会议记录转化为分钟。

In [1]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
from transformers import AutoTokenizer, pipeline

## 1. 分词器
不同的模型以不同的方式分割相同的文本。分词器很小（没有模型权重），
所以这会立即运行。 （使用非门控模型 - 无需 Hugging Face 登录。）

In [2]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
text = "Tokenization turns text into integers a model can process."
for name in ["gpt2", "bert-base-uncased", "Qwen/Qwen2.5-0.5B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    ids = tok.encode(text)
    print(f"{name:<28} {len(ids):>3} tokens  ->  {tok.convert_ids_to_tokens(ids)[:8]} ...")

gpt2                          11 tokens  ->  ['Token', 'ization', 'Ġturns', 'Ġtext', 'Ġinto', 'Ġintegers', 'Ġa', 'Ġmodel'] ...
bert-base-uncased             13 tokens  ->  ['[CLS]', 'token', '##ization', 'turns', 'text', 'into', 'integers', 'a'] ...


Qwen/Qwen2.5-0.5B-Instruct    11 tokens  ->  ['Token', 'ization', 'Ġturns', 'Ġtext', 'Ġinto', 'Ġintegers', 'Ġa', 'Ġmodel'] ...


## 2. 开源模型的会议纪要
本地摘要管道（开放权重、CPU）将记录压缩为几分钟。

In [3]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
transcript = """
Alex: Welcome everyone. The goal today is to lock the launch plan for the mobile app.
Maria: Beta testing is done. We found two minor bugs, both fixed and verified yesterday.
Sam: Marketing is ready. We'll send the announcement email on Friday morning.
Alex: Great. Let's ship to the app stores Thursday so Friday's email points to a live app.
Maria: I'll submit the build Thursday and monitor the review queue.
Sam: I'll prepare the social posts and schedule them for Friday.
Alex: Perfect. We meet again Monday to review launch metrics.
"""

summarizer = pipeline("summarization", model="Falconsai/text_summarization")
minutes = summarizer(transcript, max_length=80, min_length=25, do_sample=False)[0]["summary_text"]
print("MEETING MINUTES\n" + "-" * 14 + "\n" + minutes)

Device set to use cpu


Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MEETING MINUTES
--------------
Alex: Great. Let's ship to the app stores Thursday so Friday's email points to a live app . Sam: Marketing is ready. We'll send the announcement email on Friday morning .


**完整版（Colab GPU）：** 使用`openai/whisper`转录真实音频，然后生成
带有量化“meta-llama/Llama-3.1-8B-Instruct”的结构化会议记录——参见第 5 天。